# IntentGate: intent classification with abstention

Classify supported English requests, but defer when the model score is too low. We compare word TF-IDF logistic regression, word + character TF-IDF logistic regression, and calibrated linear SVM. The aim is to measure the tradeoff between useful automatic routing and unsupported requests being accepted by mistake. No downstream action is executed.

## 1. Data and experimental contract

[CLINC150](https://github.com/clinc/oos-eval) is a crowdsourced, English, single-intent benchmark from [Larson et al. (2019)](https://aclanthology.org/D19-1131/), licensed [CC BY 3.0](https://creativecommons.org/licenses/by/3.0/). It is not a production ticket stream. Its full version contains 150 supported intents and separate out-of-scope (OOS) examples.

Training fits features and classifiers. Validation selects both the model and rejection threshold. Test results are reported only after these decisions are frozen. OOS training examples are deliberately unused: OOS is detected using the maximum supported-class score, not learned as a single catch-all intent.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / "intent_gate").exists():
    ROOT = ROOT.parent
assert (ROOT / "intent_gate").is_dir()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
import pandas as pd
from IPython.display import display, Image
from intent_gate.data import load_data, audit
from intent_gate.train import run
from intent_gate.inference import Router

## 2. Data quality before modeling

Inspect split sizes, class balance, input lengths, duplicate rows and cross-split text overlaps. The source contains a few normalized duplicates between official splits. We retain the official splits for comparability, disclose the overlaps, and report a train-overlap-excluded test accuracy as a sensitivity check. We do not edit training data based on test labels.

In [ ]:
data = load_data()
quality = audit(data)
display(pd.DataFrame(quality["splits"]).T)
display(pd.Series({k:v for k,v in quality["cross_split_normalized_overlap"].items() if v}, name="shared normalized texts"))
print("Conflicting normalized texts:", quality["conflicting_normalized_texts"])
print(json.loads((ROOT / "outputs/provenance.json").read_text()))

In [ ]:
display(pd.DataFrame(data["train"][:8], columns=["text", "intent"]))
lengths = pd.DataFrame({"words": [len(t.split()) for t,_ in data["train"]]})
display(lengths.describe())

## 3. Baselines and preprocessing

**Word logistic regression** is the simple lexical baseline: unigram and bigram TF-IDF with sublinear term frequencies. **Word + character logistic regression** adds character 3–5-grams to share evidence across word variants. **Calibrated SVM** uses a linear margin classifier with sigmoid calibration in three training-only folds. The entire feature pipeline lives inside calibration folds to avoid vocabulary/IDF leakage into those folds.

Sparse vectors avoid dense padding and a GPU dependency. There is no oversampling: supported training classes are balanced. TF-IDF handles normalization; there is no need to standardize each sparse feature. These are CPU NLP models, not LLMs or neural embeddings. A transformer comparison is future work, not an implemented feature.

## 4. Selection policy and metrics

For each model, select a threshold that maximizes the fraction of correctly routed validation requests subject to at most 5% empirical false acceptance on the 100 OOS validation examples. Ties prefer fewer accepted mistakes, then a stricter threshold. Select the model by validation correctly routed fraction, with in-scope accuracy as tie-breaker.

Report intent accuracy and macro-F1 before rejection; OOS AUROC; in-scope coverage; accepted in-scope accuracy; OOS false-acceptance rate; and mixed accepted-request accuracy including OOS mistakes. A 5% validation constraint is not a production guarantee. Maximum class probability is only a decision score; it is not a guaranteed probability of correctness.

In [ ]:
# Full reproducible experiment: downloads if absent, trains all models, locks
# validation policies, evaluates test, saves artifacts and figures.
results = run()

## 5. Results after freezing the policy

The model is not picked by the following test table. `selection.json` is written before test reporting. Training durations and batch inference timings are measured locally, not service latency promises.

In [ ]:
report = json.loads((ROOT / "outputs/metrics.json").read_text())
print("Validation-selected model:", report["selected_model"])
columns = ["intent_accuracy", "intent_macro_f1", "oos_auroc", "in_scope_coverage", "selective_accuracy", "oos_false_accept_rate", "mixed_selective_accuracy", "nonoverlap_intent_accuracy"]
table = pd.DataFrame({name:r["test"] for name,r in report["models"].items()}).T
display(table[columns].round(4))
display(Image(filename=str(ROOT / "outputs/figures/eda.png")))
display(Image(filename=str(ROOT / "outputs/figures/tradeoffs.png")))
display(Image(filename=str(ROOT / "outputs/figures/oos_roc.png")))

## 6. Error analysis and diagnostic curves

Inspect confident mistakes, including unsupported requests accepted as supported intents. The saved coverage curve sweeps thresholds on test for diagnosis only: it must not be used to quietly choose a new deployment threshold. Supported-intent selective accuracy excludes OOS examples; mixed selective accuracy counts them as mistakes. This distinction prevents a flattering but incomplete success claim.

In [ ]:
display(pd.read_csv(ROOT / "outputs/high_confidence_errors.csv").head(10))
display(pd.read_csv(ROOT / "outputs/confusion_pairs.csv").head(10))
selected = report["models"][report["selected_model"]]["test"]
accepted = round(selected["in_scope_coverage"] * len(data["test"]))
correct = round(selected["selective_accuracy"] * accepted)
false_accepts = round(selected["oos_false_accept_rate"] * len(data["oos_test"]))
print(f"Supported accepted: {accepted}; correct: {correct}; accepted mistakes: {accepted-correct}")
print("Unsupported accepted:", false_accepts, "/", len(data["oos_test"]))

## 7. Local inference

The artifact packages the fitted feature pipeline, estimator, model name and threshold. Only load artifacts from a trusted training run: joblib deserialization can execute code. The application returns a suggested intent or abstention; it does not authenticate users, create support tickets or execute financial actions.

In [ ]:
router = Router()
for text in ["what is my account balance", "write a poem about a dragon on the moon"]:
    print(text)
    display(router.route(text))

## 8. Interpretation and limitations

Use the validation-selected model for this benchmark, but validate again on the intended domain before deployment. CLINC150 is single-intent English, not multilingual or conversational context. Short lexical models can miss paraphrases and be overconfident on shared keywords. Validation OOS coverage is small, and future unsupported requests may differ substantially. Exact duplicates are disclosed, but near-duplicate leakage is not exhaustively measured.

The next experiments would compare pretrained sentence embeddings and a fine-tuned encoder, then measure cost, latency and OOS detection at the same validation policy. Do not claim those improvements without running them. Deployment proposals—authorization, review queues, privacy-safe monitoring and rollback—are documented separately in `docs/operating-design.md`.

## ML-practice checklist

- Real published dataset and license attribution: **applied**.
- EDA, class balance, missing inputs and duplicate audit: **applied**.
- Official held-out splits and validation-only selection: **applied**, with source-overlap caveat.
- Baseline and two alternatives: **applied**.
- Train-only feature fitting and fold-local calibration: **applied**.
- Oversampling and numerical feature standardization: **not needed** for this balanced sparse-text task.
- GPU training / LLM calls: **not used**.
- Selective prediction, OOS metrics and confident-error analysis: **applied**.
- Real production traffic, load tests and authorization integrations: **not implemented**.
- Notebook rerun and automated tests: run the documented commands; results must be verified, not inferred from this checklist.